In [62]:
import pandas as pd

df = pd.read_csv("numeros-tva-6a9dbf50da6b4741045153.csv")
df_eu = pd.read_csv("./code-eu.csv", engine="python", on_bad_lines="skip")

display(df.head(1))
print("<<<<<>>>>>")
display(df_eu.head(1))

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie
0,1,Nord Technologies GmbH,DK,DK73224645,2025-07-27,portail_client


<<<<<>>>>>


,Nom français (forme courte),Date d'adhésion,Code (ISO 3166),Nom local (forme courte),Capitale,Langue(s) officielle(s),Monnaie
0,Allemagne,25 mars 1957,DE,Deutschland,Berlin,allemand (de),euro (EUR)


## nombre de sources

In [63]:
# Liste et comptage des valeurs uniques dans la colonne "source_saisie"
source_counts = df["source_saisie"].value_counts(dropna=False)

print("Comptage des valeurs uniques :")
print(source_counts)

print("\nValeurs uniques :")
print(sorted(df["source_saisie"].dropna().unique().tolist()))

print("\nNombre de valeurs uniques :", len(sorted(df["source_saisie"].dropna().unique().tolist())))

Comptage des valeurs uniques :
source_saisie
reprise_erp           2071
crm                   2019
portail_client        1990
saisie_manuelle       1976
import_fournisseur    1944
Name: count, dtype: int64

Valeurs uniques :
['crm', 'import_fournisseur', 'portail_client', 'reprise_erp', 'saisie_manuelle']

Nombre de valeurs uniques : 5


## nombre de pays

In [64]:
# Liste et comptage des valeurs uniques dans la colonne "pays_declare"
country_counts = df["pays_declare"].value_counts(dropna=False)

print("Comptage des valeurs uniques pour les pays :")
print(country_counts)

print("\nValeurs uniques des pays :")
print(sorted(df["pays_declare"].dropna().unique().tolist()))

print("\nNombre de pays uniques :", len(sorted(df["pays_declare"].dropna().unique().tolist())))

Comptage des valeurs uniques pour les pays :
pays_declare
FR    974
DK    961
BE    959
LU    956
SE    949
PT    947
NL    947
IT    947
PL    939
FI    902
ZZ    115
QQ    109
GB    104
UK    104
XX     87
Name: count, dtype: int64

Valeurs uniques des pays :
['BE', 'DK', 'FI', 'FR', 'GB', 'IT', 'LU', 'NL', 'PL', 'PT', 'QQ', 'SE', 'UK', 'XX', 'ZZ']

Nombre de pays uniques : 15


### nettoyage pays

In [66]:
# Valeurs uniques dans les 3 listes demandées
codes_eu_set = set(codes_eu.astype(str).str.upper())
pays_uniques = set(df["pays_declare"].dropna().astype(str).str.upper().unique())

dans_df_eu = sorted(pays_uniques & codes_eu_set)
hors_df_eu_sauf_uk_gb = sorted((pays_uniques - codes_eu_set) - {"UK", "GB"})
uk_gb = sorted(pays_uniques & {"UK", "GB"})

print("Dans df_eu :", dans_df_eu)
print("Hors df_eu mais ni UK ni GB :", hors_df_eu_sauf_uk_gb)
print("UK et GB :", uk_gb)

Dans df_eu : ['BE', 'DK', 'FI', 'FR', 'IT', 'LU', 'NL', 'PL', 'PT', 'SE']
Hors df_eu mais ni UK ni GB : ['QQ', 'XX', 'ZZ']
UK et GB : ['GB', 'UK']


## nombre de valeurs vides ou aberrantes

In [67]:
# Comptage des lignes contenant au moins une valeur vide/aberrante dans n'importe quelle colonne de df
df_str = df.astype("string")
df_norm = df_str.apply(lambda s: s.str.strip().str.lower())

mask_empty = df_str.eq("")                 # ""
mask_one_space = df_str.eq(" ")            # " "
mask_missing = df.isna()                   # NULL / vide
mask_na_text = df_norm.eq("n/a")           # "N/A" (insensible à la casse + espaces)
mask_null_text = df_norm.eq("null")        # "null"
mask_dash = df_norm.eq("-")                # "-"

resultats = {
    '""': int(mask_empty.any(axis=1).sum()),
    '" "': int(mask_one_space.any(axis=1).sum()),
    "NULL/vide": int(mask_missing.any(axis=1).sum()),
    '"N/A"': int(mask_na_text.any(axis=1).sum()),
    '"null"': int(mask_null_text.any(axis=1).sum()),
    '"-"': int(mask_dash.any(axis=1).sum()),
}

total_lignes_aberrantes = int(
    (mask_empty | mask_one_space | mask_missing | mask_na_text | mask_null_text | mask_dash)
    .any(axis=1)
    .sum()
)

print("Nombre de lignes contenant chaque type de valeur vide/aberrante :")
for k, v in resultats.items():
    print(f"- {k} : {v}")

print(f"\nTotal de lignes avec au moins une valeur vide/aberrante : {total_lignes_aberrantes}")

Nombre de lignes contenant chaque type de valeur vide/aberrante :
- "" : 0
- " " : 59
- NULL/vide : 146
- "N/A" : 0
- "null" : 0
- "-" : 55

Total de lignes avec au moins une valeur vide/aberrante : 260


## nombre de ligne avec un code TVA aberrant

In [70]:
# Comptage des codes TVA aberrants dans la colonne "numero_tva"
tva = df["numero_tva"].astype("string")

# 1) Vide : NaN ou chaîne vide ""
mask_vide = tva.isna() | tva.eq("")

# 2) Blanc : uniquement des espaces
mask_blanc = tva.notna() & tva.str.fullmatch(r"\s+", na=False)

# 3) Caractères non autorisés
# Autorisés : lettres A-Z/a-z, chiffres, espace, point, tiret
# Tout autre caractère est rejeté, y compris "/" et "_"
mask_non_autorise = (
    tva.notna()
    & ~mask_vide
    & ~mask_blanc
    & tva.str.contains(r"[^A-Za-z0-9 .-]", regex=True, na=False)
)

# Total lignes aberrantes
mask_total_aberrant = mask_vide | mask_blanc | mask_non_autorise

print("Nombre de lignes avec code TVA aberrant :")
print(f"- Caractères non autorisés : {int(mask_non_autorise.sum())}")
print(f"- Vide : {int(mask_vide.sum())}")
print(f"- Blanc (espaces uniquement) : {int(mask_blanc.sum())}")
print(f"- Total : {int(mask_total_aberrant.sum())}")

# Optionnel : afficher les valeurs invalides
invalid_tva = df.loc[mask_non_autorise, ["numero_tva", "pays_declare"]]
display(invalid_tva.head(20))

Nombre de lignes avec code TVA aberrant :
- Caractères non autorisés : 0
- Vide : 146
- Blanc (espaces uniquement) : 59
- Total : 205


,numero_tva,pays_declare
